In [1]:
%pip install --upgrade pylhe>=0.9.2
%pip install --upgrade awkward>=2.6.0
%pip install --upgrade tqdm
%pip install --upgrade h5py

Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
zfit 0.26.0 requires tensorflow<2.20,>=2.16.2, which is not installed.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 51.8 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import LHE_mixer_for_git.LHE_mixer as LHE_mixer
import os
import awkward as ak
import shutil
import numpy as np
import matplotlib.pyplot as plt
import pylhe
import tqdm
import h5py
print("LHE_mixer source file", LHE_mixer.__file__)

LHE_mixer source file /eos/home-i04/c/chiw/JpsiJpsiUps/MC_samples/workspace/playground/LHE_mixer_for_git/LHE_mixer.py


In [2]:
SPS_JpsiY_list_path = "my_JY_lhe_list.txt"
SPS_JpsiY_events_ak = []
# Define cache file path based on input list
cache_file = os.path.splitext(SPS_JpsiY_list_path)[0] + "_cache.h5"

# Check if cache exists
if os.path.exists(cache_file):
    print(f"Loading data from cache: {cache_file}")
    with h5py.File(cache_file, 'r') as f:
        group = f['events']
        form_json = group.attrs['form']
        length = group.attrs['length']
        form = ak.forms.from_json(form_json)
        buffers = {name: group[name][()] for name in group.keys()}
        SPS_JpsiY_events_ak = ak.from_buffers(form, length, buffers)
    print(f"Total number of events loaded from cache: {len(SPS_JpsiY_events_ak)}")
else:
    print(f"Cache not found. Reading LHE files...")
    SPS_JpsiY_events_ak = []
    
    # Get total number of files for progress bar
    with open(SPS_JpsiY_list_path, "r") as f:
        total_files = sum(1 for _ in f)
    
    # Read files with progress bar
    with open(SPS_JpsiY_list_path, "r") as f:
        for file_path in tqdm.tqdm(f, total=total_files, desc="Loading LHE files"):
            file_path = file_path.strip()
            if not os.path.isfile(file_path):
                raise FileNotFoundError(f"File not found: {file_path}")
            file_content = pylhe.read_lhe_with_attributes(file_path)
            SPS_JpsiY_events_ak.append(pylhe.to_awkward(file_content))
    
    # Concatenate all events
    SPS_JpsiY_events_ak = ak.concatenate(SPS_JpsiY_events_ak, axis=0)
    print(f"Total number of events loaded: {len(SPS_JpsiY_events_ak)}")
    
    # Save to HDF5 cache
    print(f"Saving data to cache: {cache_file}")
    with h5py.File(cache_file, 'w') as f:
        group = f.create_group('events')
        form, length, container = ak.to_buffers(ak.to_packed(SPS_JpsiY_events_ak), container=group)
        group.attrs['form'] = form.to_json()
        group.attrs['length'] = length

    print(f"Cache saved successfully")

Loading data from cache: my_JY_lhe_list_cache.h5
Total number of events loaded from cache: 554670


In [3]:
SPS_Jpsi_list_path = "my_J_lhe_list.txt"
SPS_Jpsi_events_ak = []
# Define cache file path based on input list
cache_file = os.path.splitext(SPS_Jpsi_list_path)[0] + "_cache.h5"

# Check if cache exists
if os.path.exists(cache_file):
    print(f"Loading data from cache: {cache_file}")
    with h5py.File(cache_file, 'r') as f:
        group = f['events']
        form_json = group.attrs['form']
        length = group.attrs['length']
        form = ak.forms.from_json(form_json)
        buffers = {name: group[name][()] for name in group.keys()}
        SPS_Jpsi_events_ak = ak.from_buffers(form, length, buffers)
    print(f"Total number of events loaded from cache: {len(SPS_Jpsi_events_ak)}")
else:
    print(f"Cache not found. Reading LHE files...")
    SPS_Jpsi_events_ak = []
    
    # Get total number of files for progress bar
    with open(SPS_Jpsi_list_path, "r") as f:
        total_files = sum(1 for _ in f)
    
    # Read files with progress bar
    with open(SPS_Jpsi_list_path, "r") as f:
        for file_path in tqdm.tqdm(f, total=total_files, desc="Loading LHE files"):
            file_path = file_path.strip()
            if not os.path.isfile(file_path):
                raise FileNotFoundError(f"File not found: {file_path}")
            file_content = pylhe.read_lhe_with_attributes(file_path)
            SPS_Jpsi_events_ak.append(pylhe.to_awkward(file_content))
    
    # Concatenate all events
    SPS_Jpsi_events_ak = ak.concatenate(SPS_Jpsi_events_ak, axis=0)
    print(f"Total number of events loaded: {len(SPS_Jpsi_events_ak)}")
    
    # Save to HDF5 cache
    print(f"Saving data to cache: {cache_file}")
    with h5py.File(cache_file, 'w') as f:
        group = f.create_group('events')
        form, length, container = ak.to_buffers(ak.to_packed(SPS_Jpsi_events_ak), container=group)
        group.attrs['form'] = form.to_json()
        group.attrs['length'] = length

    print(f"Cache saved successfully")

Loading data from cache: my_J_lhe_list_cache.h5
Total number of events loaded from cache: 377275


Mixing: SPS-$J/\psi$ adding SPS-$J/\psi + \Upsilon$ to obtain DPS-$J/\psi+(J/\psi+\Upsilon)$

In [4]:
DPS_JpsiY_Jpsi_events_ak = LHE_mixer.lhe_event_ak_mixer(
    [SPS_Jpsi_events_ak, SPS_JpsiY_events_ak],
    [1, 1],
    [377275, 554670],
    sort_particles_by_status=False
)

Saving in batches of 1000 events

In [5]:
output_dir = '/eos/user/c/chiw/JpsiJpsiUps/MC_samples/LHE/DPS-Jpsi-JpsiY/filter_JpsiPtMin4p0_YPtMin6p0/'
example_file = '/eos/user/c/chiw/JpsiJpsiUps/MC_samples/LHE/SPS-JpsiY/filter_YAbsMin3p0_JpsiPtMin4p0_YPtMin6p0/SPS_JpsiY_HO_filter_YAbsMin3p0_JpsiPtMin4p0_YPtMin6p0_101.lhe'

for i in tqdm.tqdm(range(0, len(DPS_JpsiY_Jpsi_events_ak), 1000), desc="Saving batches"):
    batch_events = DPS_JpsiY_Jpsi_events_ak[i:i+1000]
    batch_file = f"DPS_JpsiY_Jpsi_events_batch_{i//1000}.lhe"
    pylhe.write_lhe_file_path(
        LHE_mixer.lhe_ak_to_lhe_file(batch_events, pylhe.read_lhe_init(example_file)),
        os.path.join(output_dir, batch_file)
    )

Saving batches: 100%|██████████| 378/378 [14:27<00:00,  2.29s/it]
